## Imports

In [1]:
import random
from pathlib import Path
import json

## Configuration

In [2]:
# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Split ratios
TRAIN_RATIO = 0.8
DEV_RATIO = 0.1
TEST_RATIO = 0.1

# Language pairs to split
language_pairs = [
    "en-tl",   # baseline + distant
    "bik-tl",  # similar donor
    "en-hil",  # baseline + distant
    "msb-hil", # similar donor
    "en-war",  # baseline + distant
    "hil-war"  # similar donor
]

print(f"Random seed: {RANDOM_SEED}")
print(f"Split ratios: Train={TRAIN_RATIO*100}%, Dev={DEV_RATIO*100}%, Test={TEST_RATIO*100}%")
print(f"Total language pairs: {len(language_pairs)}")

Random seed: 42
Split ratios: Train=80.0%, Dev=10.0%, Test=10.0%
Total language pairs: 6


## Load Parallel Corpora

In [3]:
parallel_dir = Path("../data/parallel")

# Load corpus data for each pair
corpus_data = {}

for pair in language_pairs:
    src_code, tgt_code = pair.split("-")
    pair_dir = parallel_dir / pair
    
    # Read source file
    src_file = pair_dir / f"{pair}.{src_code}"
    with open(src_file, "r", encoding="utf-8") as f:
        src_lines = [line.strip() for line in f.readlines()]
    
    # Read target file
    tgt_file = pair_dir / f"{pair}.{tgt_code}"
    with open(tgt_file, "r", encoding="utf-8") as f:
        tgt_lines = [line.strip() for line in f.readlines()]
    
    # Store as parallel sentence pairs
    corpus_data[pair] = {
        "src": src_lines,
        "tgt": tgt_lines,
        "src_code": src_code,
        "tgt_code": tgt_code,
        "total": len(src_lines)
    }
    
    print(f"Loaded {pair}: {len(src_lines):,} parallel sentences")

print(f"\n✓ All {len(corpus_data)} language pairs loaded")

Loaded en-tl: 2,948 parallel sentences
Loaded bik-tl: 2,948 parallel sentences
Loaded en-hil: 2,948 parallel sentences
Loaded msb-hil: 2,948 parallel sentences
Loaded en-war: 2,948 parallel sentences
Loaded hil-war: 2,948 parallel sentences

✓ All 6 language pairs loaded


## Create Deterministic Split Indices

In [4]:
# Get total number of sentences (should be same for all pairs)
total_sentences = corpus_data[language_pairs[0]]["total"]

# Verify all pairs have same size
for pair in language_pairs:
    assert corpus_data[pair]["total"] == total_sentences, f"Size mismatch for {pair}"

print(f"Total sentences per corpus: {total_sentences:,}")

# Create shuffled indices
indices = list(range(total_sentences))
random.shuffle(indices)

# Calculate split sizes
train_size = int(total_sentences * TRAIN_RATIO)
dev_size = int(total_sentences * DEV_RATIO)
test_size = total_sentences - train_size - dev_size  # remaining goes to test

# Split indices
train_indices = indices[:train_size]
dev_indices = indices[train_size:train_size + dev_size]
test_indices = indices[train_size + dev_size:]

print(f"\nSplit sizes:")
print(f"  Train: {len(train_indices):,} sentences ({len(train_indices)/total_sentences*100:.1f}%)")
print(f"  Dev:   {len(dev_indices):,} sentences ({len(dev_indices)/total_sentences*100:.1f}%)")
print(f"  Test:  {len(test_indices):,} sentences ({len(test_indices)/total_sentences*100:.1f}%)")
print(f"  Total: {len(train_indices) + len(dev_indices) + len(test_indices):,} sentences")

Total sentences per corpus: 2,948

Split sizes:
  Train: 2,358 sentences (80.0%)
  Dev:   294 sentences (10.0%)
  Test:  296 sentences (10.0%)
  Total: 2,948 sentences


## Split and Save Data

In [5]:
def save_split(src_lines, tgt_lines, indices, output_dir, split_name, src_code, tgt_code, pair_name):
    """
    Save a data split (train/dev/test) to files.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract sentences at specified indices
    src_split = [src_lines[i] for i in indices]
    tgt_split = [tgt_lines[i] for i in indices]
    
    # Save source file
    src_file = output_dir / f"{split_name}.{src_code}"
    with open(src_file, "w", encoding="utf-8") as f:
        for line in src_split:
            f.write(f"{line}\n")
    
    # Save target file
    tgt_file = output_dir / f"{split_name}.{tgt_code}"
    with open(tgt_file, "w", encoding="utf-8") as f:
        for line in tgt_split:
            f.write(f"{line}\n")
    
    return len(src_split)

# Create splits directory
splits_dir = Path("../data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)

print("Creating splits for all language pairs...")
print("=" * 80)

split_stats = []

for pair in language_pairs:
    print(f"\nProcessing {pair}...")
    
    data = corpus_data[pair]
    pair_dir = splits_dir / pair
    
    # Save train split
    train_count = save_split(
        data["src"], data["tgt"], train_indices, 
        pair_dir, "train", data["src_code"], data["tgt_code"], pair
    )
    
    # Save dev split
    dev_count = save_split(
        data["src"], data["tgt"], dev_indices, 
        pair_dir, "dev", data["src_code"], data["tgt_code"], pair
    )
    
    # Save test split
    test_count = save_split(
        data["src"], data["tgt"], test_indices, 
        pair_dir, "test", data["src_code"], data["tgt_code"], pair
    )
    
    split_stats.append({
        "pair": pair,
        "train": train_count,
        "dev": dev_count,
        "test": test_count,
        "total": train_count + dev_count + test_count
    })
    
    print(f"  ✓ Train: {train_count:,} | Dev: {dev_count:,} | Test: {test_count:,}")

print("\n" + "=" * 80)
print("✓ All splits created successfully!")

Creating splits for all language pairs...

Processing en-tl...
  ✓ Train: 2,358 | Dev: 294 | Test: 296

Processing bik-tl...
  ✓ Train: 2,358 | Dev: 294 | Test: 296

Processing en-hil...
  ✓ Train: 2,358 | Dev: 294 | Test: 296

Processing msb-hil...
  ✓ Train: 2,358 | Dev: 294 | Test: 296

Processing en-war...
  ✓ Train: 2,358 | Dev: 294 | Test: 296

Processing hil-war...
  ✓ Train: 2,358 | Dev: 294 | Test: 296

✓ All splits created successfully!


## Save Split Metadata

In [6]:
# Save split metadata
split_metadata = {
    "description": "Train/Dev/Test splits for MT training",
    "random_seed": RANDOM_SEED,
    "split_ratios": {
        "train": TRAIN_RATIO,
        "dev": DEV_RATIO,
        "test": TEST_RATIO
    },
    "total_sentences": total_sentences,
    "split_sizes": {
        "train": len(train_indices),
        "dev": len(dev_indices),
        "test": len(test_indices)
    },
    "language_pairs": split_stats
}

metadata_file = splits_dir / "split_metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(split_metadata, f, indent=2)

print(f"✓ Metadata saved to {metadata_file}")

# Save indices for reproducibility
indices_data = {
    "random_seed": RANDOM_SEED,
    "train_indices": train_indices,
    "dev_indices": dev_indices,
    "test_indices": test_indices
}

indices_file = splits_dir / "split_indices.json"
with open(indices_file, "w", encoding="utf-8") as f:
    json.dump(indices_data, f, indent=2)

print(f"✓ Split indices saved to {indices_file}")

✓ Metadata saved to ..\data\splits\split_metadata.json
✓ Split indices saved to ..\data\splits\split_indices.json


## Verify Splits

In [7]:
# Verify no overlap between splits
train_set = set(train_indices)
dev_set = set(dev_indices)
test_set = set(test_indices)

assert len(train_set & dev_set) == 0, "Train and dev overlap!"
assert len(train_set & test_set) == 0, "Train and test overlap!"
assert len(dev_set & test_set) == 0, "Dev and test overlap!"

print("✓ No overlap between splits - verified!")

# Show sample from a split
sample_pair = "en-tl"
pair_dir = splits_dir / sample_pair
src_code, tgt_code = sample_pair.split("-")

# Read first 3 sentences from train split
train_src = pair_dir / f"train.{src_code}"
train_tgt = pair_dir / f"train.{tgt_code}"

with open(train_src, "r", encoding="utf-8") as f:
    src_samples = [line.strip() for line in f.readlines()[:3]]

with open(train_tgt, "r", encoding="utf-8") as f:
    tgt_samples = [line.strip() for line in f.readlines()[:3]]

print(f"\nSample from {sample_pair.upper()} train split:")
print("=" * 100)
for i, (src, tgt) in enumerate(zip(src_samples, tgt_samples), 1):
    print(f"\n{i}. {src_code.upper()}: {src}")
    print(f"   {tgt_code.upper()}: {tgt}")
print("=" * 100)

✓ No overlap between splits - verified!

Sample from EN-TL train split:

1. EN: His lord said unto him, Well done, good and faithful servant: thou hast been faithful over a few things, I will set thee over many things; enter thou into the joy of thy lord.
   TL: Sinabi sa kanya ng panginoon niya, ‘Magaling! Mabuti at tapat na alipin. Naging tapat ka sa kaunting bagay, pamamahalain kita sa maraming bagay. Pumasok ka sa kagalakan ng iyong panginoon.’

2. EN: and shall cut him asunder, and appoint his portion with the hypocrites: there shall be the weeping and the gnashing of teeth.
   TL: at siya'y pagpuputul-putulin at ilalagay na kasama ng mga mapagkunwari, kung saan ay magkakaroon ng pagtangis at pagngangalit ng mga ngipin.

3. EN: And when the ten heard it, they were moved with indignation concerning the two brethren.
   TL: Nang marinig ito ng sampu ay nagalit sila sa dalawang magkapatid.
